# DSAA1001 Purchase & Redemption Data Project - Phase 2

This notebook completes Phase 2 of the Purchase & Redemption data project.

Main outputs generated by this notebook:

- `fund.db`
- `example_train.csv`
- `example_test.csv`

Main tasks:

1. Clean missing values and abnormal negative values.
2. Store cleaned data into SQLite and run SQL queries.
3. Create indexes and compare query running time before and after indexing.
4. Generate daily-level train/test CSV files for Phase 3.
5. Use aggregation queries to support two data-analysis conclusions.

In [1]:
import sys
import sqlite3
import time
from pathlib import Path

import pandas as pd
import numpy as np

print("Python version:", sys.version)
print("pandas version:", pd.__version__)
print("numpy version:", np.__version__)
print("sqlite3 is available")

Python version: 3.11.15 | packaged by Anaconda, Inc. | (main, Jun 11 2026, 15:12:53) [MSC v.1942 64 bit (AMD64)]
pandas version: 3.0.3
numpy version: 2.4.4
sqlite3 is available


## 1. Project Setup and File Check

The raw dataset folder should be named `Purchase Redemption Data` and placed in the same folder as this notebook.

In [2]:
PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "Purchase Redemption Data"
DB_PATH = PROJECT_DIR / "fund.db"
TRAIN_PATH = PROJECT_DIR / "example_train.csv"
TEST_PATH = PROJECT_DIR / "example_test.csv"

required_files = [
    "user_balance_table.csv",
    "user_profile_table.csv",
    "mfd_day_share_interest.csv",
    "mfd_bank_shibor.csv",
    "comp_predict_table.csv",
]

print("Current working directory:", PROJECT_DIR)
print("Dataset folder:", DATA_DIR)
print("Dataset folder exists:", DATA_DIR.exists())

missing_files = []
for file_name in required_files:
    file_path = DATA_DIR / file_name
    exists = file_path.exists()
    print(f"{file_name} exists: {exists}")
    if not exists:
        missing_files.append(file_name)

if missing_files:
    raise FileNotFoundError(
        "Missing required dataset files: " + ", ".join(missing_files) +
        "\nPlease make sure the folder is named exactly 'Purchase Redemption Data' "
        "and is in the same directory as this notebook."
    )

Current working directory: e:\Documents\SummerSchool\Intro to DSA\DSAA1001_Purchase_Redemption_Project
Dataset folder: e:\Documents\SummerSchool\Intro to DSA\DSAA1001_Purchase_Redemption_Project\Purchase Redemption Data
Dataset folder exists: True
user_balance_table.csv exists: True
user_profile_table.csv exists: True
mfd_day_share_interest.csv exists: True
mfd_bank_shibor.csv exists: True
comp_predict_table.csv exists: True


## 2. Load Raw CSV Tables

For Phase 2 database management, the four cleaned tables are:

- `user_balance_table`
- `user_profile_table`
- `mfd_day_share_interest`
- `mfd_bank_shibor`

`comp_predict_table.csv` only contains target prediction dates, so it is checked above but not stored as a main database table in this phase.

In [3]:
table_files = {
    "user_balance_table": "user_balance_table.csv",
    "user_profile_table": "user_profile_table.csv",
    "mfd_day_share_interest": "mfd_day_share_interest.csv",
    "mfd_bank_shibor": "mfd_bank_shibor.csv",
}

raw_tables = {}

for table_name, file_name in table_files.items():
    file_path = DATA_DIR / file_name
    df = pd.read_csv(file_path, low_memory=False)
    df.columns = [str(c).strip().lower() for c in df.columns]
    raw_tables[table_name] = df
    print("=" * 80)
    print(table_name, "shape:", df.shape)
    display(df.head())

user_balance_table shape: (2840421, 18)


,user_id,report_date,tbalance,ybalance,total_purchase_amt,direct_purchase_amt,purchase_bal_amt,purchase_bank_amt,total_redeem_amt,consume_amt,transfer_amt,tftobal_amt,tftocard_amt,share_amt,category1,category2,category3,category4
0,1,20140805,20385,20383,2,0,0,0,0,0,0,0,0,2,NaN,NaN,NaN,NaN
1,1,20140808,20391,20389,2,0,0,0,0,0,0,0,0,2,NaN,NaN,NaN,NaN
2,1,20140811,20397,20395,2,0,0,0,0,0,0,0,0,2,NaN,NaN,NaN,NaN
3,1,20140814,20403,20401,2,0,0,0,0,0,0,0,0,2,NaN,NaN,NaN,NaN
4,1,20140817,20409,20407,2,0,0,0,0,0,0,0,0,2,NaN,NaN,NaN,NaN


user_profile_table shape: (28041, 4)


,user_id,sex,city,constellation
0,2,1,6411949,狮子座
1,12,1,6412149,摩羯座
2,22,1,6411949,双子座
3,23,1,6411949,双鱼座
4,25,1,6481949,双鱼座


mfd_day_share_interest shape: (427, 3)


,mfd_date,mfd_daily_yield,mfd_7daily_yield
0,20130701,1.5787,6.307
1,20130702,1.5461,6.174
2,20130703,1.4670,6.034
3,20130704,1.4223,5.903
4,20130705,1.3845,5.739


mfd_bank_shibor shape: (294, 9)


,mfd_date,interest_o_n,interest_1_w,interest_2_w,interest_1_m,interest_3_m,interest_6_m,interest_9_m,interest_1_y
0,20130701,4.456,5.423,6.040,6.8800,5.295,4.2390,4.2820,4.4125
1,20130702,3.786,4.750,5.074,5.8000,5.211,4.2344,4.2808,4.4070
2,20130703,3.400,4.242,4.658,5.2000,5.148,4.2300,4.2796,4.4022
3,20130704,3.348,3.938,4.464,5.1020,5.029,4.2287,4.2776,4.4000
4,20130705,3.380,3.816,4.295,4.7885,4.939,4.2273,4.2749,4.4000


## 3. Task 1: Data Cleaning

Cleaning rules used here:

- Strings equal to `unknown` are treated as missing values.
- Blank strings are treated as missing values.
- For `category1`, `category2`, `category3`, and `category4`, missing or unknown values are filled with `0`.
- Other numeric missing values are filled with the column median. If the whole column is missing, `0` is used.
- Categorical missing values are filled with the most frequent value.
- Negative values in numeric columns are treated as invalid and replaced with `0`.

After cleaning, the notebook checks that there are no missing values, no literal `unknown` values, and no negative numeric values.

In [4]:
def count_unknown_strings(df: pd.DataFrame) -> int:
    """Count cells that are string values equal to 'unknown', ignoring case and spaces."""
    total = 0
    for col in df.columns:
        if pd.api.types.is_object_dtype(df[col]) or pd.api.types.is_string_dtype(df[col]):
            s = df[col].astype("string").str.strip().str.lower()
            total += int((s == "unknown").sum())
    return total


def count_negative_numbers(df: pd.DataFrame) -> int:
    """Count negative values in numeric columns."""
    total = 0
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        total += int((df[col] < 0).sum())
    return total


def infer_numeric_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Convert object columns to numeric only if all non-missing values can be converted."""
    df = df.copy()
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            continue
        converted = pd.to_numeric(df[col], errors="coerce")
        non_missing = df[col].notna()
        if converted[non_missing].notna().all():
            df[col] = converted
    return df


def clean_dataframe(df: pd.DataFrame, table_name: str) -> pd.DataFrame:
    df = df.copy()
    df.columns = [str(c).strip().lower() for c in df.columns]

    # Replace explicit missing markers with NaN.
    object_cols = df.select_dtypes(include=["object", "string"]).columns
    for col in object_cols:
        df[col] = df[col].replace(r"^\s*unknown\s*$", np.nan, regex=True)
        df[col] = df[col].replace(r"^\s*$", np.nan, regex=True)

    # According to the data description hint, UNKNOWN in category1-category4 means 0.
    category_cols = ["category1", "category2", "category3", "category4"]
    for col in category_cols:
        if col in df.columns:
            df[col] = df[col].fillna(0)

    # Convert numeric-looking columns after unknown values have been handled.
    df = infer_numeric_columns(df)

    # Fill remaining missing values.
    for col in df.columns:
        if df[col].isna().sum() == 0:
            continue

        if pd.api.types.is_numeric_dtype(df[col]):
            if col in category_cols:
                fill_value = 0
            else:
                fill_value = df[col].median(skipna=True)
                if pd.isna(fill_value):
                    fill_value = 0
            df[col] = df[col].fillna(fill_value)
        else:
            mode_values = df[col].dropna().mode()
            fill_value = mode_values.iloc[0] if len(mode_values) > 0 else "missing"
            df[col] = df[col].fillna(fill_value)

    # Replace invalid negative numeric values with 0.
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        neg_mask = df[col] < 0
        if neg_mask.any():
            df.loc[neg_mask, col] = 0

    return df


cleaning_summary_rows = []
cleaned_tables = {}

for table_name, df in raw_tables.items():
    before_missing = int(df.isna().sum().sum())
    before_unknown = count_unknown_strings(df)
    before_negative = count_negative_numbers(infer_numeric_columns(df.replace(r"^\s*unknown\s*$", np.nan, regex=True)))

    cleaned_df = clean_dataframe(df, table_name)
    cleaned_tables[table_name] = cleaned_df

    after_missing = int(cleaned_df.isna().sum().sum())
    after_unknown = count_unknown_strings(cleaned_df)
    after_negative = count_negative_numbers(cleaned_df)

    cleaning_summary_rows.append({
        "table": table_name,
        "rows": cleaned_df.shape[0],
        "columns": cleaned_df.shape[1],
        "missing_before": before_missing,
        "unknown_before": before_unknown,
        "negative_before": before_negative,
        "missing_after": after_missing,
        "unknown_after": after_unknown,
        "negative_after": after_negative,
    })

cleaning_summary = pd.DataFrame(cleaning_summary_rows)
display(cleaning_summary)

assert cleaning_summary["missing_after"].sum() == 0, "There are still missing values after cleaning."
assert cleaning_summary["unknown_after"].sum() == 0, "There are still 'unknown' strings after cleaning."
assert cleaning_summary["negative_after"].sum() == 0, "There are still negative numeric values after cleaning."

print("Task 1 complete: no missing values, no 'unknown' strings, and no negative numeric values remain after cleaning.")

,table,rows,columns,missing_before,unknown_before,negative_before,missing_after,unknown_after,negative_after
0,user_balance_table,2840421,18,10666728,0,0,0,0,0
1,user_profile_table,28041,4,0,0,0,0,0,0
2,mfd_day_share_interest,427,3,0,0,0,0,0,0
3,mfd_bank_shibor,294,9,0,0,0,0,0,0


Task 1 complete: no missing values, no 'unknown' strings, and no negative numeric values remain after cleaning.


## 4. Task 2: Store Cleaned Tables into SQLite

The cleaned four tables are stored in a SQLite database named `fund.db`.

In [5]:
# Recreate the database from scratch so that timing comparisons are not affected by old indexes.
if DB_PATH.exists():
    DB_PATH.unlink()

conn = sqlite3.connect(DB_PATH)

for table_name, df in cleaned_tables.items():
    print(f"Writing {table_name} to SQLite, shape={df.shape} ...")
    df.to_sql(table_name, conn, if_exists="replace", index=False, chunksize=50_000)

conn.commit()

row_count_rows = []
for table_name in table_files.keys():
    count = pd.read_sql_query(f"SELECT COUNT(*) AS row_count FROM {table_name}", conn)["row_count"].iloc[0]
    row_count_rows.append({"table": table_name, "row_count": int(count)})

row_counts = pd.DataFrame(row_count_rows)
display(row_counts)
print("Database created:", DB_PATH)

Writing user_balance_table to SQLite, shape=(2840421, 18) ...
Writing user_profile_table to SQLite, shape=(28041, 4) ...
Writing mfd_day_share_interest to SQLite, shape=(427, 3) ...
Writing mfd_bank_shibor to SQLite, shape=(294, 9) ...


,table,row_count
0,user_balance_table,2840421
1,user_profile_table,28041
2,mfd_day_share_interest,427
3,mfd_bank_shibor,294


Database created: e:\Documents\SummerSchool\Intro to DSA\DSAA1001_Purchase_Redemption_Project\fund.db


## 5. Task 2: SQL Queries Before Indexing

Required SQL queries:

1. Query records from `user_balance_table` for one specific date.
2. Query records from `user_balance_table` for one specific user.
3. Join `user_balance_table` and `user_profile_table` for users from one specific city on one specific date.

The specific date, user ID, and city are selected automatically from valid values in the dataset.

In [6]:
def quote_identifier(name: str) -> str:
    """Safely quote a SQLite identifier such as a column name."""
    return '"' + str(name).replace('"', '""') + '"'


def get_columns(conn: sqlite3.Connection, table_name: str):
    info = pd.read_sql_query(f"PRAGMA table_info({table_name})", conn)
    return info["name"].tolist()


def to_python_value(value):
    """Convert numpy scalar values to normal Python values for sqlite parameters."""
    if hasattr(value, "item"):
        return value.item()
    return value


# Basic required-column check.
user_balance_cols = set(get_columns(conn, "user_balance_table"))
user_profile_cols = set(get_columns(conn, "user_profile_table"))

required_balance_cols = {"user_id", "report_date", "total_purchase_amt", "total_redeem_amt"}
required_profile_cols = {"user_id", "city"}

missing_balance_cols = required_balance_cols - user_balance_cols
missing_profile_cols = required_profile_cols - user_profile_cols

if missing_balance_cols:
    raise ValueError(f"Missing columns in user_balance_table: {missing_balance_cols}")
if missing_profile_cols:
    raise ValueError(f"Missing columns in user_profile_table: {missing_profile_cols}")

# Choose a valid date with many records.
specific_date = pd.read_sql_query(
    """
    SELECT report_date
    FROM user_balance_table
    GROUP BY report_date
    ORDER BY COUNT(*) DESC
    LIMIT 1
    """,
    conn,
)["report_date"].iloc[0]

# Choose a valid user with many records.
specific_user = pd.read_sql_query(
    """
    SELECT user_id
    FROM user_balance_table
    GROUP BY user_id
    ORDER BY COUNT(*) DESC
    LIMIT 1
    """,
    conn,
)["user_id"].iloc[0]

# Choose a city-date pair that exists in the join.
city_date_df = pd.read_sql_query(
    """
    SELECT p.city AS city, b.report_date AS report_date, COUNT(*) AS n_records
    FROM user_balance_table b
    JOIN user_profile_table p
        ON b.user_id = p.user_id
    GROUP BY p.city, b.report_date
    ORDER BY n_records DESC
    LIMIT 1
    """,
    conn,
)

specific_city = city_date_df["city"].iloc[0]
specific_city_date = city_date_df["report_date"].iloc[0]

specific_date = to_python_value(specific_date)
specific_user = to_python_value(specific_user)
specific_city = to_python_value(specific_city)
specific_city_date = to_python_value(specific_city_date)

print("Chosen date for Query 1:", specific_date)
print("Chosen user_id for Query 2:", specific_user)
print("Chosen city and date for Query 3:", specific_city, specific_city_date)

Chosen date for Query 1: 20140831
Chosen user_id for Query 2: 27988
Chosen city and date for Query 3: 6281949 20140831


In [7]:
def run_timed_query(conn, sql: str, params=None, repeats: int = 3):
    """Run a SQL query several times and return the result plus timing statistics."""
    if params is None:
        params = ()
    times = []
    result = None
    for _ in range(repeats):
        start = time.perf_counter()
        result = pd.read_sql_query(sql, conn, params=params)
        elapsed = time.perf_counter() - start
        times.append(elapsed)
    return result, min(times), sum(times) / len(times)


sql_query_1 = """
SELECT user_id, report_date, total_purchase_amt, total_redeem_amt
FROM user_balance_table
WHERE report_date = ?
ORDER BY user_id
"""

sql_query_2 = """
SELECT user_id, report_date, total_purchase_amt, total_redeem_amt
FROM user_balance_table
WHERE user_id = ?
ORDER BY report_date
"""

sql_query_3 = """
SELECT b.user_id,
       b.report_date,
       p.city AS City,
       b.total_purchase_amt,
       b.total_redeem_amt
FROM user_balance_table b
JOIN user_profile_table p
    ON b.user_id = p.user_id
WHERE p.city = ? AND b.report_date = ?
ORDER BY b.user_id
"""

queries = [
    ("Query 1: one specific date", sql_query_1, (specific_date,)),
    ("Query 2: one specific user", sql_query_2, (specific_user,)),
    ("Query 3: one city on one date", sql_query_3, (specific_city, specific_city_date)),
]

timing_before_rows = []
query_results_before = {}

for query_name, sql, params in queries:
    result, min_time, avg_time = run_timed_query(conn, sql, params=params, repeats=3)
    query_results_before[query_name] = result
    timing_before_rows.append({
        "query": query_name,
        "rows_returned": len(result),
        "min_time_before_index_sec": min_time,
        "avg_time_before_index_sec": avg_time,
    })
    print("=" * 80)
    print(query_name)
    print("Rows returned:", len(result))
    display(result.head(10))

timing_before = pd.DataFrame(timing_before_rows)
display(timing_before)

Query 1: one specific date
Rows returned: 12614


,user_id,report_date,total_purchase_amt,total_redeem_amt
0,1,20140831,2,0
1,3,20140831,0,0
2,6,20140831,0,0
3,7,20140831,0,0
4,8,20140831,2971,2800
5,10,20140831,12,0
6,11,20140831,123,30351
7,15,20140831,200,0
8,17,20140831,0,0
9,18,20140831,1,0


Query 2: one specific user
Rows returned: 427


,user_id,report_date,total_purchase_amt,total_redeem_amt
0,27988,20130701,315,0
1,27988,20130702,309,0
2,27988,20130703,293,0
3,27988,20130704,284,200
4,27988,20130705,277,200
5,27988,20130706,273,0
6,27988,20130707,273,0
7,27988,20130708,262,0
8,27988,20130709,262,0
9,27988,20130710,256,0


Query 3: one city on one date
Rows returned: 3230


,user_id,report_date,City,total_purchase_amt,total_redeem_amt
0,7,20140831,6281949,0,0
1,17,20140831,6281949,0,0
2,18,20140831,6281949,1,0
3,31,20140831,6281949,170,0
4,36,20140831,6281949,0,0
5,44,20140831,6281949,0,0
6,57,20140831,6281949,31,0
7,71,20140831,6281949,12,0
8,75,20140831,6281949,2,0
9,105,20140831,6281949,10,0


,query,rows_returned,min_time_before_index_sec,avg_time_before_index_sec
0,Query 1: one specific date,12614,0.759004,0.838843
1,Query 2: one specific user,427,0.657529,0.768214
2,Query 3: one city on one date,3230,0.662135,0.822716


## 6. Task 2: Create Indexes and Compare Query Time

Indexes are created on the columns suggested by the project instruction:

- `user_balance_table(report_date)`
- `user_balance_table(user_id)`
- `user_profile_table(city, user_id)`

In [8]:
index_statements = [
    "CREATE INDEX IF NOT EXISTS idx_user_balance_report_date ON user_balance_table(report_date)",
    "CREATE INDEX IF NOT EXISTS idx_user_balance_user_id ON user_balance_table(user_id)",
    "CREATE INDEX IF NOT EXISTS idx_user_profile_city_user_id ON user_profile_table(city, user_id)",
]

for stmt in index_statements:
    conn.execute(stmt)
conn.commit()

print("Indexes created.")

# Confirm indexes.
indexes = pd.read_sql_query(
    """
    SELECT name, tbl_name, sql
    FROM sqlite_master
    WHERE type = 'index'
    ORDER BY tbl_name, name
    """,
    conn,
)
display(indexes)

Indexes created.


,name,tbl_name,sql
0,idx_user_balance_report_date,user_balance_table,CREATE INDEX idx_user_balance_report_date ON u...
1,idx_user_balance_user_id,user_balance_table,CREATE INDEX idx_user_balance_user_id ON user_...
2,idx_user_profile_city_user_id,user_profile_table,CREATE INDEX idx_user_profile_city_user_id ON ...


In [9]:
timing_after_rows = []
query_results_after = {}

for query_name, sql, params in queries:
    result, min_time, avg_time = run_timed_query(conn, sql, params=params, repeats=3)
    query_results_after[query_name] = result
    timing_after_rows.append({
        "query": query_name,
        "rows_returned_after": len(result),
        "min_time_after_index_sec": min_time,
        "avg_time_after_index_sec": avg_time,
    })

timing_after = pd.DataFrame(timing_after_rows)

timing_comparison = timing_before.merge(timing_after, on="query", how="inner")
timing_comparison["min_speedup_ratio"] = (
    timing_comparison["min_time_before_index_sec"] / timing_comparison["min_time_after_index_sec"]
)
timing_comparison["avg_speedup_ratio"] = (
    timing_comparison["avg_time_before_index_sec"] / timing_comparison["avg_time_after_index_sec"]
)

display(timing_comparison)
print("Task 2 complete: SQL query results and query time comparison before/after indexing are shown above.")

,query,rows_returned,min_time_before_index_sec,avg_time_before_index_sec,rows_returned_after,min_time_after_index_sec,avg_time_after_index_sec,min_speedup_ratio,avg_speedup_ratio
0,Query 1: one specific date,12614,0.759004,0.838843,12614,0.066570,0.152982,11.401659,5.483276
1,Query 2: one specific user,427,0.657529,0.768214,427,0.001126,0.001232,583.795256,623.550620
2,Query 3: one city on one date,3230,0.662135,0.822716,3230,0.101413,0.126617,6.529063,6.497656


Task 2 complete: SQL query results and query time comparison before/after indexing are shown above.


## 7. Task 3: Generate Daily-Level Train/Test CSV Files

This section uses SQL JOIN statements to combine:

- `user_balance_table`
- `mfd_day_share_interest`
- `mfd_bank_shibor`

The generated daily-level data include:

- `report_date`
- daily total purchase amount
- daily total redeem amount
- useful money-market yield columns
- useful Shibor / interest-rate columns

Then the daily-level data are split into:

- `example_train.csv`: dates before `20140801`
- `example_test.csv`: dates from `20140801` to `20140831`

In [10]:
interest_cols = [c for c in get_columns(conn, "mfd_day_share_interest") if c != "mfd_date"]
shibor_cols = [c for c in get_columns(conn, "mfd_bank_shibor") if c != "mfd_date"]

select_parts = [
    "d.report_date",
    "d.daily_total_purchase_amt",
    "d.daily_total_redeem_amt",
]

for col in interest_cols:
    select_parts.append(f"i.{quote_identifier(col)} AS {quote_identifier(col)}")

for col in shibor_cols:
    select_parts.append(f"s.{quote_identifier(col)} AS {quote_identifier(col)}")

select_clause = ",\n       ".join(select_parts)

sql_daily_level = f"""
WITH daily_balance AS (
    SELECT report_date,
           SUM(total_purchase_amt) AS daily_total_purchase_amt,
           SUM(total_redeem_amt) AS daily_total_redeem_amt
    FROM user_balance_table
    GROUP BY report_date
)
SELECT {select_clause}
FROM daily_balance d
LEFT JOIN mfd_day_share_interest i
    ON d.report_date = i.mfd_date
LEFT JOIN mfd_bank_shibor s
    ON d.report_date = s.mfd_date
ORDER BY d.report_date
"""

print(sql_daily_level)

daily_df = pd.read_sql_query(sql_daily_level, conn)

daily_df["report_date"] = pd.to_numeric(daily_df["report_date"], errors="coerce").astype(int)
daily_df = daily_df.sort_values("report_date").reset_index(drop=True)

# Fill possible missing values caused by dates not matched in the joined interest-rate tables.
feature_cols = [
    c for c in daily_df.columns
    if c not in ["report_date", "daily_total_purchase_amt", "daily_total_redeem_amt"]
]

if feature_cols:
    daily_df[feature_cols] = daily_df[feature_cols].ffill().bfill()

# Final safety fill if any missing values remain.
for col in daily_df.columns:
    if daily_df[col].isna().sum() == 0:
        continue
    if pd.api.types.is_numeric_dtype(daily_df[col]):
        daily_df[col] = daily_df[col].fillna(0)
    else:
        mode_values = daily_df[col].dropna().mode()
        daily_df[col] = daily_df[col].fillna(mode_values.iloc[0] if len(mode_values) else "missing")

print("Daily-level dataset shape:", daily_df.shape)
display(daily_df.head())
display(daily_df.tail())

example_train = daily_df[daily_df["report_date"] < 20140801].copy()
example_test = daily_df[
    (daily_df["report_date"] >= 20140801) &
    (daily_df["report_date"] <= 20140831)
].copy()

example_train.to_csv(TRAIN_PATH, index=False)
example_test.to_csv(TEST_PATH, index=False)

print("example_train.csv shape:", example_train.shape)
print("example_test.csv shape:", example_test.shape)
print("Saved:", TRAIN_PATH)
print("Saved:", TEST_PATH)

display(example_train.head())
display(example_test.head())

assert TRAIN_PATH.exists(), "example_train.csv was not created."
assert TEST_PATH.exists(), "example_test.csv was not created."
print("Task 3 complete: example_train.csv and example_test.csv were generated.")


WITH daily_balance AS (
    SELECT report_date,
           SUM(total_purchase_amt) AS daily_total_purchase_amt,
           SUM(total_redeem_amt) AS daily_total_redeem_amt
    FROM user_balance_table
    GROUP BY report_date
)
SELECT d.report_date,
       d.daily_total_purchase_amt,
       d.daily_total_redeem_amt,
       i."mfd_daily_yield" AS "mfd_daily_yield",
       i."mfd_7daily_yield" AS "mfd_7daily_yield",
       s."interest_o_n" AS "interest_o_n",
       s."interest_1_w" AS "interest_1_w",
       s."interest_2_w" AS "interest_2_w",
       s."interest_1_m" AS "interest_1_m",
       s."interest_3_m" AS "interest_3_m",
       s."interest_6_m" AS "interest_6_m",
       s."interest_9_m" AS "interest_9_m",
       s."interest_1_y" AS "interest_1_y"
FROM daily_balance d
LEFT JOIN mfd_day_share_interest i
    ON d.report_date = i.mfd_date
LEFT JOIN mfd_bank_shibor s
    ON d.report_date = s.mfd_date
ORDER BY d.report_date

Daily-level dataset shape: (427, 13)


,report_date,daily_total_purchase_amt,daily_total_redeem_amt,mfd_daily_yield,mfd_7daily_yield,interest_o_n,interest_1_w,interest_2_w,interest_1_m,interest_3_m,interest_6_m,interest_9_m,interest_1_y
0,20130701,32488348,5525022,1.5787,6.307,4.456,5.423,6.040,6.8800,5.295,4.2390,4.2820,4.4125
1,20130702,29037390,2554548,1.5461,6.174,3.786,4.750,5.074,5.8000,5.211,4.2344,4.2808,4.4070
2,20130703,27270770,5953867,1.4670,6.034,3.400,4.242,4.658,5.2000,5.148,4.2300,4.2796,4.4022
3,20130704,18321185,6410729,1.4223,5.903,3.348,3.938,4.464,5.1020,5.029,4.2287,4.2776,4.4000
4,20130705,11648749,2763587,1.3845,5.739,3.380,3.816,4.295,4.7885,4.939,4.2273,4.2749,4.4000


,report_date,daily_total_purchase_amt,daily_total_redeem_amt,mfd_daily_yield,mfd_7daily_yield,interest_o_n,interest_1_w,interest_2_w,interest_1_m,interest_3_m,interest_6_m,interest_9_m,interest_1_y
422,20140827,302194801,468164147,1.1079,4.123,2.861,3.349,3.9828,4.178,4.6680,4.8729,4.9365,5.0
423,20140828,245082751,297893861,1.1043,4.116,2.876,3.538,4.0500,4.133,4.6689,4.8727,4.9373,5.0
424,20140829,267554713,273756380,1.1169,4.123,2.908,3.672,4.1990,4.114,4.6659,4.8726,4.9364,5.0
425,20140830,199708772,196374134,1.1206,4.134,2.908,3.672,4.1990,4.114,4.6659,4.8726,4.9364,5.0
426,20140831,275090213,292943033,1.1204,4.146,2.908,3.672,4.1990,4.114,4.6659,4.8726,4.9364,5.0


example_train.csv shape: (396, 13)
example_test.csv shape: (31, 13)
Saved: e:\Documents\SummerSchool\Intro to DSA\DSAA1001_Purchase_Redemption_Project\example_train.csv
Saved: e:\Documents\SummerSchool\Intro to DSA\DSAA1001_Purchase_Redemption_Project\example_test.csv


,report_date,daily_total_purchase_amt,daily_total_redeem_amt,mfd_daily_yield,mfd_7daily_yield,interest_o_n,interest_1_w,interest_2_w,interest_1_m,interest_3_m,interest_6_m,interest_9_m,interest_1_y
0,20130701,32488348,5525022,1.5787,6.307,4.456,5.423,6.040,6.8800,5.295,4.2390,4.2820,4.4125
1,20130702,29037390,2554548,1.5461,6.174,3.786,4.750,5.074,5.8000,5.211,4.2344,4.2808,4.4070
2,20130703,27270770,5953867,1.4670,6.034,3.400,4.242,4.658,5.2000,5.148,4.2300,4.2796,4.4022
3,20130704,18321185,6410729,1.4223,5.903,3.348,3.938,4.464,5.1020,5.029,4.2287,4.2776,4.4000
4,20130705,11648749,2763587,1.3845,5.739,3.380,3.816,4.295,4.7885,4.939,4.2273,4.2749,4.4000


,report_date,daily_total_purchase_amt,daily_total_redeem_amt,mfd_daily_yield,mfd_7daily_yield,interest_o_n,interest_1_w,interest_2_w,interest_1_m,interest_3_m,interest_6_m,interest_9_m,interest_1_y
396,20140801,374884735,252540858,1.1173,4.171,3.196,3.907,4.047,4.210,4.7149,4.900,4.9558,5.0
397,20140802,189092130,172250225,1.1180,4.168,3.196,3.907,4.047,4.210,4.7149,4.900,4.9558,5.0
398,20140803,173825397,127125217,1.1178,4.166,3.196,3.907,4.047,4.210,4.7149,4.900,4.9558,5.0
399,20140804,330640884,322907524,1.1451,4.176,3.182,3.842,4.086,4.105,4.7081,4.900,4.9551,5.0
400,20140805,394780870,221706539,1.1203,4.178,3.145,3.738,3.998,4.056,4.7049,4.899,4.9543,5.0


Task 3 complete: example_train.csv and example_test.csv were generated.


## 8. Task 4: Aggregation-Based Data Analysis

This section uses SQL aggregation queries to support two conclusions:

1. User behavior differs across cities.
2. Purchase and redemption amounts fluctuate across dates.

Note: amount values in this dataset are measured in cents.

In [11]:
# Analysis 1: Compare purchase and redeem behavior across cities.
city_analysis_sql = """
SELECT p.city AS City,
       COUNT(DISTINCT b.user_id) AS n_users,
       COUNT(*) AS n_records,
       SUM(b.total_purchase_amt) AS total_purchase_amt,
       SUM(b.total_redeem_amt) AS total_redeem_amt,
       AVG(b.total_purchase_amt) AS avg_purchase_per_record,
       AVG(b.total_redeem_amt) AS avg_redeem_per_record
FROM user_balance_table b
JOIN user_profile_table p
    ON b.user_id = p.user_id
GROUP BY p.city
ORDER BY total_purchase_amt DESC
LIMIT 10
"""

city_analysis = pd.read_sql_query(city_analysis_sql, conn)
display(city_analysis)

if not city_analysis.empty:
    top_purchase_city = city_analysis.loc[city_analysis["total_purchase_amt"].idxmax()]
    top_redeem_city = city_analysis.loc[city_analysis["total_redeem_amt"].idxmax()]

    print("Conclusion 1:")
    print(
        f"Among the top cities shown, City {top_purchase_city['City']} has the largest total purchase amount "
        f"({top_purchase_city['total_purchase_amt']:.0f} cents). "
        f"City {top_redeem_city['City']} has the largest total redeem amount "
        f"({top_redeem_city['total_redeem_amt']:.0f} cents). "
        "This suggests that purchase and redemption behavior differs across user-city groups."
    )

,City,n_users,n_records,total_purchase_amt,total_redeem_amt,avg_purchase_per_record,avg_redeem_per_record
0,6281949,6307,763073,28768484939,22427115574,37700.829330,29390.524333
1,6301949,3243,402330,14872381039,11844359355,36965.627815,29439.413802
2,6081949,4003,384263,13988521454,10572108653,36403.508675,27512.689624
3,6412149,4791,452641,13172303031,10409211355,29100.994013,22996.616204
4,6411949,4803,430040,11680071946,9495976436,27160.431462,22081.612027
5,6481949,2929,245117,6320575805,4907341103,25785.954483,20020.402922
6,6581949,1965,162957,3788745242,3062185094,23249.969268,18791.368852


Conclusion 1:
Among the top cities shown, City 6281949.0 has the largest total purchase amount (28768484939 cents). City 6281949.0 has the largest total redeem amount (22427115574 cents). This suggests that purchase and redemption behavior differs across user-city groups.


In [12]:
# Analysis 2: Daily purchase and redemption trend.
daily_trend_sql = """
SELECT report_date,
       SUM(total_purchase_amt) AS daily_total_purchase_amt,
       SUM(total_redeem_amt) AS daily_total_redeem_amt,
       SUM(total_purchase_amt) - SUM(total_redeem_amt) AS daily_net_purchase_amt
FROM user_balance_table
GROUP BY report_date
ORDER BY report_date
"""

daily_trend = pd.read_sql_query(daily_trend_sql, conn)
daily_trend["report_date"] = pd.to_numeric(daily_trend["report_date"], errors="coerce").astype(int)

display(daily_trend.head())
display(daily_trend.tail())

print("Top 5 purchase dates:")
display(daily_trend.sort_values("daily_total_purchase_amt", ascending=False).head(5))

print("Top 5 redeem dates:")
display(daily_trend.sort_values("daily_total_redeem_amt", ascending=False).head(5))

if not daily_trend.empty:
    max_purchase_day = daily_trend.loc[daily_trend["daily_total_purchase_amt"].idxmax()]
    max_redeem_day = daily_trend.loc[daily_trend["daily_total_redeem_amt"].idxmax()]
    avg_purchase = daily_trend["daily_total_purchase_amt"].mean()
    avg_redeem = daily_trend["daily_total_redeem_amt"].mean()

    print("Conclusion 2:")
    print(
        f"The highest daily purchase amount occurs on {int(max_purchase_day['report_date'])} "
        f"with {max_purchase_day['daily_total_purchase_amt']:.0f} cents, while the highest daily redeem amount "
        f"occurs on {int(max_redeem_day['report_date'])} with {max_redeem_day['daily_total_redeem_amt']:.0f} cents. "
        f"Across all available dates, the average daily purchase amount is {avg_purchase:.0f} cents and "
        f"the average daily redeem amount is {avg_redeem:.0f} cents. "
        "The daily totals show clear fluctuations over time, so Phase 3 should use date-level features rather than assuming demand is constant."
    )

,report_date,daily_total_purchase_amt,daily_total_redeem_amt,daily_net_purchase_amt
0,20130701,32488348,5525022,26963326
1,20130702,29037390,2554548,26482842
2,20130703,27270770,5953867,21316903
3,20130704,18321185,6410729,11910456
4,20130705,11648749,2763587,8885162


,report_date,daily_total_purchase_amt,daily_total_redeem_amt,daily_net_purchase_amt
422,20140827,302194801,468164147,-165969346
423,20140828,245082751,297893861,-52811110
424,20140829,267554713,273756380,-6201667
425,20140830,199708772,196374134,3334638
426,20140831,275090213,292943033,-17852820


Top 5 purchase dates:


,report_date,daily_total_purchase_amt,daily_total_redeem_amt,daily_net_purchase_amt
212,20140129,952479658,215164666,737314992
211,20140128,843424742,248334316,595090426
225,20140211,818803205,243058162,575745043
226,20140212,763009770,354585919,408423851
231,20140217,706118060,405809267,300308793


Top 5 redeem dates:


,report_date,daily_total_purchase_amt,daily_total_redeem_amt,daily_net_purchase_amt
359,20140625,264663201,547295931,-282632730
245,20140303,505305862,513017360,-7711498
351,20140617,270350693,502560223,-232209530
253,20140311,430500816,496039886,-65539070
241,20140227,672909288,492786036,180123252


Conclusion 2:
The highest daily purchase amount occurs on 20140129 with 952479658 cents, while the highest daily redeem amount occurs on 20140625 with 547295931 cents. Across all available dates, the average daily purchase amount is 216840945 cents and the average daily redeem amount is 170300463 cents. The daily totals show clear fluctuations over time, so Phase 3 should use date-level features rather than assuming demand is constant.


## 9. Final Readiness Check

This final cell checks that all required Phase 2 output files were generated successfully.

In [13]:
final_outputs = [DB_PATH, TRAIN_PATH, TEST_PATH]
final_check = []

for path in final_outputs:
    final_check.append({
        "file": path.name,
        "exists": path.exists(),
        "size_bytes": path.stat().st_size if path.exists() else 0,
    })

final_check_df = pd.DataFrame(final_check)
display(final_check_df)

assert final_check_df["exists"].all(), "Some required output files are missing."

print("Phase 2 complete.")
print("Generated files:")
for path in final_outputs:
    print("-", path.name)

conn.close()

,file,exists,size_bytes
0,fund.db,True,181477376
1,example_train.csv,True,34819
2,example_test.csv,True,3004


Phase 2 complete.
Generated files:
- fund.db
- example_train.csv
- example_test.csv
